In [ ]:
import os, sys, requests
from pathlib import Path
from urllib.parse import urlparse
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [ ]:
# Cria a conexão Spark

# 1. Remove qualquer barreira de proxy local que jogue o tráfego para a rede da empresa
os.environ.pop('HTTP_PROXY', None)
os.environ.pop('HTTPS_PROXY', None)
os.environ.pop('http_proxy', None)
os.environ.pop('https_proxy', None)

# 2. Garante que o Spark use o Python correto do venv
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# 3. Força o IP local estrito
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

# 4. Inicializa configurando a autenticação local do Worker
spark = SparkSession.builder \
    .appName("TesteLocal") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.network.auth.enabled", "false") \
    .getOrCreate()

In [ ]:
df_munic_brasil_lat_long = \
    spark.read.csv("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\municipios_brasil_lat_long.csv"
                  ,header=True
                  ,inferSchema=True)

df_temp_mensal = \
    spark.read.csv("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\tb_temperatura_mensal.csv"
                  ,header=True
                  ,inferSchema=True
                  ,sep=";")

In [ ]:
# Arredondar as colunas de latitude e longitude para 4 casas decimais, ficará mais próximo dos dados do ERA5, que é de 25 em 25 graus
df_munic_brasil_lat_long = \
    (df_munic_brasil_lat_long
        .withColumn("latitude", F.round(df_munic_brasil_lat_long["latitude"], 2))
        .withColumn("longitude", F.round(df_munic_brasil_lat_long["longitude"], 2)))

df_munic_brasil_lat_long_grades = \
    (df_munic_brasil_lat_long
        .withColumn("lat_grade", F.round(F.col("latitude") * 4) / 4)
        .withColumn("lon_grade", F.round(F.col("longitude") * 4) / 4)
    )

df_munic_brasil_lat_long_grades.printSchema()
df_munic_brasil_lat_long_grades.limit(10).show(truncate=False)

# df_temp_mensal.printSchema()
# df_temp_mensal.limit(10).show(truncate=False)

In [46]:
# print("Número de registros do DataFrame de municípios: ", df_munic_brasil_lat_long_grades.count())
# print("Número de registros do DataFrame de temperatura mensal: ", df_temp_mensal.count())

df_munic_brasil_lat_long_grades.printSchema()
df_temp_mensal.printSchema()

root
 |-- codigo_municipio: integer (nullable = true)
 |-- nome_municipio: string (nullable = true)
 |-- codigo_estado: integer (nullable = true)
 |-- uf: string (nullable = true)
 |-- nome_estado: string (nullable = true)
 |-- codigo_regiao: integer (nullable = true)
 |-- nome_regiao: string (nullable = true)
 |-- ano: integer (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- lat_grade: double (nullable = true)
 |-- lon_grade: double (nullable = true)

root
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- temp_min_mes: double (nullable = true)
 |-- temp_max_mes: double (nullable = true)
 |-- temp_media_mes: double (nullable = true)
 |-- percentil_05_mes: double (nullable = true)
 |-- percentil_90_mes: double (nullable = true)



In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_ibge = df_munic_brasil_lat_long
df_era5 = df_temp_mensal


# --- CONFIGURAÇÃO ---
RAIO_BUSCA_GRAUS = 0.4 

# 1. RENOMEAR COLUNAS DO IBGE PARA EVITAR DUPLICIDADE
df_ibge_renomeado = \
    (df_ibge
        .withColumnRenamed("latitude" , "lat_ibge")
        .withColumnRenamed("longitude", "lon_ibge")
        .withColumnRenamed("ano"      , "ano_ibge")
    )

# 2. EXECUTAR O JOIN COM AS COLUNAS JÁ DISTINTAS
df_cruzado = \
    (df_era5.join(F.broadcast(df_ibge_renomeado)
        ,on=[df_era5["ano"] == df_ibge_renomeado["ano_ibge"]
           ,df_era5["latitude"].between(df_ibge_renomeado["lat_ibge"] - RAIO_BUSCA_GRAUS, df_ibge_renomeado["lat_ibge"] + RAIO_BUSCA_GRAUS)
           ,df_era5["longitude"].between(df_ibge_renomeado["lon_ibge"] - RAIO_BUSCA_GRAUS, df_ibge_renomeado["lon_ibge"] + RAIO_BUSCA_GRAUS)
           ]
        ,how="inner")
    )

# 3. CÁLCULO DA DISTÂNCIA REAL (Haversine em KM)
# Agora as referências são diretas e seguras
lat1 = F.radians(F.col("latitude"))      # Vem do ERA5
lon1 = F.radians(F.col("longitude"))     # Vem do ERA5
lat2 = F.radians(F.col("lat_ibge"))      # Vem do IBGE
lon2 = F.radians(F.col("lon_ibge"))      # Vem do IBGE

dlat = lat2 - lat1
dlon = lon2 - lon1

# Fórmula de Haversine
a = F.sin(dlat / 2)**2 + F.cos(lat1) * F.cos(lat2) * F.sin(dlon / 2)**2
c = 2 * F.atan2(F.sqrt(a), F.sqrt(1 - a))
R = 6371.0  # Raio da Terra em km

df_com_distancia = df_cruzado.withColumn("distancia_km", c * R)

# 4. SELECIONAR APENAS O VIZINHO MAIS PRÓXIMO
window_spec = \
    (Window.partitionBy("codigo_municipio"
                       ,"mes")
            .orderBy("distancia_km"))

df_resultado_final = \
    (df_com_distancia
        .withColumn("rank", F.row_number().over(window_spec))
        .filter(F.col("rank") == 1)
        .drop("rank"))


In [45]:
# Exibir o resultado limpo
(df_resultado_final
    .filter("codigo_municipio like '110012%'")
    .select("codigo_municipio"
           ,"nome_municipio"
           ,"uf"
           ,"ano"
           ,"mes"
           ,"temp_min_mes"
           ,"temp_max_mes"
           ,"temp_media_mes"
           ,"percentil_05_mes"
           ,"percentil_90_mes"
           ,F.col("latitude").alias("era5_lat")
           ,F.col("longitude").alias("era5_lon")
           ,F.col("lat_ibge").alias("ibge_lat")
           ,F.col("lon_ibge").alias("ibge_lon")
           ,F.col("temp_media_mes")
           ,F.col("distancia_km"))
        .show())

+----------------+--------------+---+----+---+------------+------------+------------------+----------------+----------------+--------+--------+--------+--------+------------------+----------------+
|codigo_municipio|nome_municipio| uf| ano|mes|temp_min_mes|temp_max_mes|    temp_media_mes|percentil_05_mes|percentil_90_mes|era5_lat|era5_lon|ibge_lat|ibge_lon|    temp_media_mes|    distancia_km|
+----------------+--------------+---+----+---+------------+------------+------------------+----------------+----------------+--------+--------+--------+--------+------------------+----------------+
|         1100122|     Ji-Paraná| RO|2024|  1|   24.711884|    28.12265|26.496709000000003|       24.723785|       27.660004|   -10.5|  -61.75|  -10.46|  -61.76|26.496709000000003|4.58022082107807|
|         1100122|     Ji-Paraná| RO|2024|  2|   25.108124|    28.16275| 26.46473106896552|        25.21463|       27.874664|   -10.5|  -61.75|  -10.46|  -61.76| 26.46473106896552|4.58022082107807|
+---------

In [ ]:
df_resultado_final.filter("temp_media_mes is not null").count() 

# 11140 Total de registros 
#  5570 Municípios 